# Custom CUDA Kernel Implementation

Implementing a specialized neural network architecture with **raw CUDA kernels** for optimal GPU performance and pharmaceutical-specific optimizations.

## Raw CUDA Kernel Features:
- **Custom embedding lookup kernels** with optimized memory coalescing for drug representation
- **Hand-optimized matrix multiplication kernels** with tiling strategies for linear layers
- **Fused activation function kernels** combining ReLU and dropout in single GPU operation
- **Custom batch normalization kernels** with drug-specific statistical computation
- **Specialized loss computation kernels** for cross-entropy and softmax operations
- **GPU memory pool management** with CuPy integration for zero-copy operations

## Technical Implementation:
- **CuPy Raw Kernels**: Direct CUDA C++ kernel compilation and execution
- **Memory Coalescing**: Optimized memory access patterns for drug embedding lookups
- **Kernel Fusion**: Combined operations to minimize GPU kernel launch overhead
- **Thread Block Optimization**: Pharmaceutical workload-specific thread configuration
- **Hybrid Architecture**: Custom CUDA acceleration with PyTorch fallback reliability

#Complete Custom CUDA Drug Interaction Model

Starting fresh with a complete, standalone Custom CUDA kernel implementation for drug interaction prediction. This section is completely self-contained and doesn't rely on any previous cells.

## What We'll Build:
1. **Complete data loading and preprocessing** from scratch
2. **Custom CUDA kernel manager** with 6 specialized kernels
3. **Custom CUDA neural network** with hybrid architecture
4. **Full training pipeline** with real pharmaceutical data
5. **Model evaluation and deployment** ready system

Let's start from the beginning with all necessary imports and setup.

In [22]:
# 🔧 Complete Fresh Setup - All Imports and Dependencies
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import cupy as cp
from cupy.cuda import memory
import time
import warnings
warnings.filterwarnings('ignore')

# Setup device and memory
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 Using device: {device}")

# Initialize CuPy memory pool for optimal GPU memory management
mempool = memory.MemoryPool(memory.malloc_managed)
cp.cuda.set_allocator(mempool.malloc)
print("✅ CuPy memory pool initialized")

# Check CUDA availability for custom kernels
CUSTOM_CUDA_AVAILABLE = torch.cuda.is_available() and cp.cuda.is_available()
print(f"🚀 Custom CUDA kernels available: {CUSTOM_CUDA_AVAILABLE}")

if CUSTOM_CUDA_AVAILABLE:
    print(f"📊 GPU: {torch.cuda.get_device_name()}")
    print(f"💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  CUDA not available, will use CPU fallbacks")

🔥 Using device: cuda
✅ CuPy memory pool initialized
🚀 Custom CUDA kernels available: True
📊 GPU: NVIDIA GeForce RTX 4070 Laptop GPU
💾 GPU Memory: 8.6 GB


In [24]:
# 📊 Fresh Data Loading - Real Drug Interaction Dataset
print("🔄 Loading fresh drug interaction dataset...")

# Load the actual drug interaction data
try:
    df = pd.read_csv('combined_dataset_final.csv')
    print(f"✅ Dataset loaded successfully!")
    print(f"📈 Total samples: {len(df):,}")
    print(f"📋 Columns: {list(df.columns)}")
    
    # Auto-detect target column (common names: label, target, interaction, safety, etc.)
    target_column_candidates = ['label', 'target', 'interaction', 'safety', 'outcome', 'result', 'class']
    target_column = None
    
    for candidate in target_column_candidates:
        if candidate in df.columns:
            target_column = candidate
            break
    
    # If no standard target column found, look for binary/categorical columns
    if target_column is None:
        for col in df.columns:
            if df[col].dtype in ['int64', 'object'] and df[col].nunique() <= 5:
                # Check if it looks like a target (binary or small categorical)
                unique_vals = df[col].unique()
                if len(unique_vals) == 2 or any(str(v).lower() in ['safe', 'unsafe', 'yes', 'no', '0', '1'] for v in unique_vals):
                    target_column = col
                    break
    
    if target_column:
        print(f"🎯 Target column detected: '{target_column}'")
        print(f"🎯 Target distribution:\n{df[target_column].value_counts()}")
        
        # Ensure target is properly encoded as 0/1
        if df[target_column].dtype == 'object':
            # Convert text labels to numeric
            unique_vals = df[target_column].unique()
            if len(unique_vals) == 2:
                # Binary classification
                label_map = {unique_vals[0]: 0, unique_vals[1]: 1}
                df['label'] = df[target_column].map(label_map)
                print(f"📝 Converted labels: {label_map}")
            else:
                # Multi-class - use label encoding
                from sklearn.preprocessing import LabelEncoder
                le = LabelEncoder()
                df['label'] = le.fit_transform(df[target_column])
                print(f"📝 Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
        else:
            # Numeric target
            df['label'] = df[target_column]
    else:
        # Create a synthetic target based on number of drugs (more drugs = higher risk)
        print("⚠️ No target column found. Creating synthetic target based on drug count...")
        drug_cols = [col for col in df.columns if 'drug' in col.lower()]
        df['drug_count'] = df[drug_cols].notna().sum(axis=1)
        # Higher risk with more drugs
        df['label'] = (df['drug_count'] > df['drug_count'].median()).astype(int)
        target_column = 'label'
        print(f"🎯 Synthetic target created based on drug count")
    
    # Display sample data
    print("\n🔍 Sample data:")
    drug_cols = [col for col in df.columns if 'drug' in col.lower()][:5]  # First 5 drug columns
    display_cols = drug_cols + ['label']
    available_cols = [col for col in display_cols if col in df.columns]
    print(df[available_cols].head())
    
except FileNotFoundError:
    print("❌ Dataset file not found! Creating synthetic data for demonstration...")
    # Create realistic synthetic drug interaction data
    np.random.seed(42)
    n_samples = 20000
    
    # Generate drug combinations with realistic patterns
    drug_names = [f"Drug_{i:03d}" for i in range(1, 501)]  # 500 unique drugs
    
    data = []
    for _ in range(n_samples):
        # Random number of drugs (2-5 per interaction)
        n_drugs = np.random.choice([2, 3, 4, 5], p=[0.4, 0.3, 0.2, 0.1])
        drugs = np.random.choice(drug_names, size=n_drugs, replace=False)
        
        # Create interaction record
        record = {f'drug_{chr(97+i)}': drug if i < len(drugs) else '' for i in range(10)}
        
        # Simulate interaction safety (more drugs = higher risk)
        base_risk = 0.15 + (n_drugs - 2) * 0.1
        record['label'] = int(np.random.random() < base_risk)
        
        data.append(record)
    
    df = pd.DataFrame(data)
    print(f"✅ Synthetic dataset created: {len(df):,} samples")

print("✅ Data loading complete!")

🔄 Loading fresh drug interaction dataset...
✅ Dataset loaded successfully!
📈 Total samples: 20,482,172
📋 Columns: ['subject_id', 'doses_per_24_hrs', 'drug1', 'drug2', 'drug3', 'drug4', 'drug5', 'drug6', 'drug7', 'drug8', 'drug9', 'drug10', 'safety_label', 'total_drugs', 'has_dosage_info', 'drug_combination_id']
✅ Dataset loaded successfully!
📈 Total samples: 20,482,172
📋 Columns: ['subject_id', 'doses_per_24_hrs', 'drug1', 'drug2', 'drug3', 'drug4', 'drug5', 'drug6', 'drug7', 'drug8', 'drug9', 'drug10', 'safety_label', 'total_drugs', 'has_dosage_info', 'drug_combination_id']
🎯 Target column detected: 'safety_label'
🎯 Target distribution:
safety_label
safe      20290631
unsafe      191541
Name: count, dtype: int64
🎯 Target column detected: 'safety_label'
🎯 Target distribution:
safety_label
safe      20290631
unsafe      191541
Name: count, dtype: int64
📝 Converted labels: {'safe': 0, 'unsafe': 1}

🔍 Sample data:
📝 Converted labels: {'safe': 0, 'unsafe': 1}

🔍 Sample data:
              

In [26]:
# 🔬 Fresh Data Preprocessing - Memory-Efficient Pipeline
class FreshDrugInteractionPreprocessor:
    """Memory-efficient data preprocessing pipeline for large drug interaction datasets"""
    
    def __init__(self, max_vocab_size=10000, sample_size=50000):
        self.drug_encoder = LabelEncoder()
        self.max_drugs = 10
        self.drug_vocab_size = None
        self.max_vocab_size = max_vocab_size  # Limit vocabulary to prevent memory issues
        self.sample_size = sample_size  # Sample dataset if too large
        
    def fit_transform(self, df):
        """Fit preprocessor and transform data with memory optimization"""
        print("🔄 Starting memory-efficient data preprocessing...")
        
        # Sample dataset if too large to prevent memory issues
        if len(df) > self.sample_size:
            print(f"⚠️  Large dataset detected ({len(df):,} samples)")
            print(f"🎯 Sampling {self.sample_size:,} samples for training")
            df_sample = df.sample(n=self.sample_size, random_state=42)
        else:
            df_sample = df.copy()
        
        print(f"📊 Processing {len(df_sample):,} samples")
        
        # Extract drug columns efficiently
        drug_cols = [col for col in df_sample.columns if 'drug' in col.lower()]
        print(f"🔍 Found {len(drug_cols)} drug columns: {drug_cols[:5]}...")
        
        # Collect unique drugs efficiently with frequency counting
        drug_frequency = {}
        for col in drug_cols:
            print(f"   Processing column: {col}")
            
            # Get non-null, non-empty values
            valid_drugs = df_sample[col].dropna()
            valid_drugs = valid_drugs[valid_drugs != '']
            
            # Count frequencies
            for drug in valid_drugs.unique():
                if drug not in drug_frequency:
                    drug_frequency[drug] = 0
                drug_frequency[drug] += len(valid_drugs[valid_drugs == drug])
        
        print(f"📚 Found {len(drug_frequency):,} unique drugs")
        
        # Select most frequent drugs to limit vocabulary size
        if len(drug_frequency) > self.max_vocab_size - 2:  # Reserve space for special tokens
            print(f"🎯 Limiting vocabulary to top {self.max_vocab_size - 2} most frequent drugs")
            sorted_drugs = sorted(drug_frequency.items(), key=lambda x: x[1], reverse=True)
            top_drugs = [drug for drug, freq in sorted_drugs[:self.max_vocab_size - 2]]
        else:
            top_drugs = list(drug_frequency.keys())
        
        # Add special tokens
        vocab_drugs = ['<PAD>', '<UNK>'] + top_drugs
        
        # Fit drug encoder with limited vocabulary
        print(f"🔧 Fitting encoder with {len(vocab_drugs):,} drugs")
        self.drug_encoder.fit(vocab_drugs)
        self.drug_vocab_size = len(vocab_drugs)
        
        print(f"✅ Vocabulary created:")
        print(f"   📚 Total vocabulary size: {self.drug_vocab_size:,}")
        print(f"   🔍 Top 5 drugs: {top_drugs[:5]}")
        print(f"   📈 Sample drug frequencies: {list(sorted(drug_frequency.items(), key=lambda x: x[1], reverse=True)[:3])}")
        
        # Transform data
        X, y = self._transform_data(df_sample, set(top_drugs))
        
        print(f"✅ Preprocessing complete!")
        print(f"📊 Input shape: {X.shape}")
        print(f"🎯 Target shape: {y.shape}")
        
        return X, y
    
    def _transform_data(self, df, valid_drugs):
        """Transform dataframe to model input format with memory efficiency"""
        # Get drug columns (limit to max_drugs)
        drug_cols = [col for col in df.columns if 'drug' in col.lower()][:self.max_drugs]
        
        # Initialize drug matrix
        drug_matrix = np.zeros((len(df), self.max_drugs), dtype=np.int32)
        
        # Get token indices
        pad_token = self.drug_encoder.transform(['<PAD>'])[0]
        unk_token = self.drug_encoder.transform(['<UNK>'])[0]
        
        print("🔄 Encoding drug interactions...")
        
        for i, col in enumerate(drug_cols):
            print(f"   Encoding column {i+1}/{len(drug_cols)}: {col}")
            
            for j, drug in enumerate(df[col].fillna('')):
                if drug == '' or pd.isna(drug):
                    drug_matrix[j, i] = pad_token
                elif drug in valid_drugs:
                    drug_matrix[j, i] = self.drug_encoder.transform([drug])[0]
                else:
                    drug_matrix[j, i] = unk_token
        
        # Extract targets
        y = df['label'].values.astype(np.int64)
        
        print("✅ Data encoding complete!")
        return drug_matrix, y

# Initialize memory-efficient preprocessor
print("🚀 Initializing memory-efficient preprocessor...")
fresh_preprocessor = FreshDrugInteractionPreprocessor(
    max_vocab_size=5000,    # Limit vocabulary to prevent memory issues
    sample_size=20000       # Sample large datasets
)

# Transform data with memory optimization
X, y = fresh_preprocessor.fit_transform(df)

# Store important parameters
drug_vocab_size = fresh_preprocessor.drug_vocab_size
max_drugs = fresh_preprocessor.max_drugs
num_classes = len(np.unique(y))

print(f"\n🎛️  Model Parameters:")
print(f"   Drug vocabulary size: {drug_vocab_size:,}")
print(f"   Max drugs per interaction: {max_drugs}")
print(f"   Number of classes: {num_classes}")
print(f"   Input shape: {X.shape}")
print(f"   Target distribution: {np.bincount(y)}")

# Memory cleanup
import gc
gc.collect()
print("🧹 Memory cleanup completed!")

🚀 Initializing memory-efficient preprocessor...
🔄 Starting memory-efficient data preprocessing...
⚠️  Large dataset detected (20,482,172 samples)
🎯 Sampling 20,000 samples for training
📊 Processing 20,000 samples
🔍 Found 12 drug columns: ['drug1', 'drug2', 'drug3', 'drug4', 'drug5']...
   Processing column: drug1
📊 Processing 20,000 samples
🔍 Found 12 drug columns: ['drug1', 'drug2', 'drug3', 'drug4', 'drug5']...
   Processing column: drug1
   Processing column: drug2
   Processing column: drug2
   Processing column: drug3
   Processing column: drug3
   Processing column: drug4
   Processing column: drug4
   Processing column: drug5
   Processing column: drug6
   Processing column: drug7
   Processing column: drug8
   Processing column: drug9
   Processing column: drug10
   Processing column: total_drugs
   Processing column: drug_combination_id
   Processing column: drug5
   Processing column: drug6
   Processing column: drug7
   Processing column: drug8
   Processing column: drug9
  